# Tiny Encoder-Decoder Training (CPU) on TinyStories

This notebook trains an ultra-small encoder-decoder Transformer (T5-style) from scratch on `Geralt-Targaryen/tinystories`.

- Hardware: CPU only
- Goal: story continuation (`prefix -> continuation`)
- Scope: tiny model for local experimentation

Run cells top to bottom.

In [ ]:
# Install dependencies (run once)
%pip install -q --upgrade pip
%pip install -q torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q transformers datasets tokenizers accelerate sentencepiece

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from typing import Dict, List, Optional, Tuple

import torch
from datasets import Dataset, load_dataset
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from transformers import (
    DataCollatorForSeq2Seq,
    PreTrainedTokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5Config,
    T5ForConditionalGeneration,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

c:\Users\jayan\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.8.0+cpu
CUDA available: False


In [3]:
# Tiny CPU-friendly config (edit if needed)
DATASET_NAME = "Geralt-Targaryen/tinystories"
MAX_SAMPLES = 8000          # start tiny for laptop CPU
TEST_SIZE = 0.02
VOCAB_SIZE = 2000
MAX_SOURCE_LEN = 96
MAX_TARGET_LEN = 96
OUTPUT_DIR = "artifacts/tiny-encdec-cpu"

# Model size
D_MODEL = 128
D_FF = 256
NUM_LAYERS = 2
NUM_DECODER_LAYERS = 2
NUM_HEADS = 4
DROPOUT = 0.1

# Training
NUM_EPOCHS = 2
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 20
EVAL_STEPS = 200
SAVE_STEPS = 200

In [4]:
def make_input_target_pair(text: str, min_words: int = 8) -> Optional[Tuple[str, str]]:
    """Create a story-continuation pair from one story.

    This function splits a story into two segments: a prefix used as encoder input and
    a suffix used as decoder target. It also prepends a task prompt to stabilize
    encoder-decoder training.

    Args:
        text (str): Raw story text.
        min_words (int): Minimum number of words required for a valid sample.

    Returns:
        Optional[Tuple[str, str]]: A tuple of (input_text, target_text) if valid,
            otherwise None.
    """
    words = text.strip().split()
    if len(words) < min_words:
        return None

    split_idx = max(1, int(len(words) * 0.4))
    prefix = " ".join(words[:split_idx])
    suffix = " ".join(words[split_idx:])
    return f"continue story: {prefix}", suffix


def prepare_text_datasets(
    dataset_name: str,
    max_samples: int,
    test_size: float = 0.02,
) -> Tuple[Dataset, Dataset, List[str]]:
    """Load TinyStories and convert it into seq2seq text datasets.

    The resulting datasets contain two columns: `input_text` and `target_text`.
    Raw texts are also returned to train a small tokenizer.

    Args:
        dataset_name (str): Hugging Face dataset identifier.
        max_samples (int): Maximum number of rows to consume from the train split.
        test_size (float): Fraction of pairs to reserve for validation.

    Returns:
        Tuple[Dataset, Dataset, List[str]]: Train text dataset, validation text dataset,
            and list of raw stories.
    """
    raw = load_dataset(dataset_name, split="train")
    raw = raw.select(range(min(max_samples, len(raw))))

    src_texts: List[str] = []
    tgt_texts: List[str] = []
    raw_texts: List[str] = []

    for row in raw:
        story = row["text"]
        pair = make_input_target_pair(story)
        if pair is None:
            continue
        src, tgt = pair
        src_texts.append(src)
        tgt_texts.append(tgt)
        raw_texts.append(story)

    paired = Dataset.from_dict({"input_text": src_texts, "target_text": tgt_texts})
    split = paired.train_test_split(test_size=test_size, seed=42)
    return split["train"], split["test"], raw_texts


def build_bpe_tokenizer(
    texts: List[str],
    vocab_size: int,
    save_path: str,
) -> PreTrainedTokenizerFast:
    """Train a compact BPE tokenizer and return HF fast tokenizer.

    A small vocabulary is used to keep model size and CPU training cost low.

    Args:
        texts (List[str]): Text corpus for tokenizer training.
        vocab_size (int): Vocabulary size for BPE.
        save_path (str): Path where tokenizer JSON will be saved.

    Returns:
        PreTrainedTokenizerFast: A tokenizer configured with required special tokens.
    """
    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.normalizer = normalizers.Sequence([normalizers.NFD(), normalizers.Lowercase()])
    tok.pre_tokenizer = pre_tokenizers.Whitespace()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<unk>", "<bos>", "</s>"],
    )
    tok.train_from_iterator(texts, trainer=trainer)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    tok.save(save_path)

    hf_tok = PreTrainedTokenizerFast(
        tokenizer_file=save_path,
        pad_token="<pad>",
        unk_token="<unk>",
        bos_token="<bos>",
        eos_token="</s>",
    )
    return hf_tok


def tokenize_seq2seq_datasets(
    train_ds: Dataset,
    val_ds: Dataset,
    tokenizer: PreTrainedTokenizerFast,
    max_source_len: int,
    max_target_len: int,
) -> Tuple[Dataset, Dataset]:
    """Tokenize text datasets for encoder-decoder training.

    This function converts source and target strings into token IDs and writes
    decoder labels in the `labels` field expected by `Seq2SeqTrainer`.

    Args:
        train_ds (Dataset): Training dataset containing input/target text.
        val_ds (Dataset): Validation dataset containing input/target text.
        tokenizer (PreTrainedTokenizerFast): Shared tokenizer for source and target.
        max_source_len (int): Maximum encoder input length.
        max_target_len (int): Maximum decoder target length.

    Returns:
        Tuple[Dataset, Dataset]: Tokenized train and validation datasets.
    """
    def _tok(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
        model_inputs = tokenizer(
            batch["input_text"],
            truncation=True,
            max_length=max_source_len,
        )
        labels = tokenizer(
            text_target=batch["target_text"],
            truncation=True,
            max_length=max_target_len,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    train_tok = train_ds.map(_tok, batched=True, remove_columns=train_ds.column_names)
    val_tok = val_ds.map(_tok, batched=True, remove_columns=val_ds.column_names)
    return train_tok, val_tok


def build_tiny_t5_model(
    vocab_size: int,
    max_positions: int,
    d_model: int,
    d_ff: int,
    num_layers: int,
    num_decoder_layers: int,
    num_heads: int,
    dropout: float,
    pad_token_id: int,
    eos_token_id: int,
) -> T5ForConditionalGeneration:
    """Construct an ultra-small T5-like encoder-decoder model.

    The architecture still includes multi-head attention, feed-forward sublayers,
    layer normalization, residual connections, and softmax output.

    Args:
        vocab_size (int): Tokenizer vocabulary size.
        max_positions (int): Max sequence length used for position settings.
        d_model (int): Hidden size of Transformer layers.
        d_ff (int): Feed-forward hidden size.
        num_layers (int): Number of encoder layers.
        num_decoder_layers (int): Number of decoder layers.
        num_heads (int): Number of attention heads.
        dropout (float): Dropout probability.
        pad_token_id (int): Padding token id.
        eos_token_id (int): End-of-sequence token id.

    Returns:
        T5ForConditionalGeneration: Initialized encoder-decoder model.
    """
    config = T5Config(
        vocab_size=vocab_size,
        d_model=d_model,
        d_ff=d_ff,
        num_layers=num_layers,
        num_decoder_layers=num_decoder_layers,
        num_heads=num_heads,
        dropout_rate=dropout,
        pad_token_id=pad_token_id,
        eos_token_id=eos_token_id,
        decoder_start_token_id=pad_token_id,
        n_positions=max_positions,
    )
    return T5ForConditionalGeneration(config)

In [5]:
train_text_ds, val_text_ds, raw_texts = prepare_text_datasets(
    dataset_name=DATASET_NAME,
    max_samples=MAX_SAMPLES,
    test_size=TEST_SIZE,
)

print(f"Train text pairs: {len(train_text_ds)}")
print(f"Val text pairs:   {len(val_text_ds)}")
print("Example input:", train_text_ds[0]["input_text"][:180])
print("Example target:", train_text_ds[0]["target_text"][:180])

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating train split: 100%|██████████| 4939657/4939657 [00:10<00:00, 469031.48 examples/s]


Train text pairs: 7840
Val text pairs:   160
Example input: continue story: Once upon a time, a little boy named Timmy went to the zoo with his mom. They walked around and saw many animals. Suddenly, Timmy saw a big crocodile! He was scared
Example target: then, they saw a man carrying a sack. Timmy asked "Mommy, what's in the sack?" His mom said "I don't know, Timmy. Maybe it's some food for the animals." But then, they heard the ma


In [6]:
tokenizer = build_bpe_tokenizer(
    texts=raw_texts,
    vocab_size=VOCAB_SIZE,
    save_path=f"{OUTPUT_DIR}/tokenizer.json",
)

train_tok, val_tok = tokenize_seq2seq_datasets(
    train_ds=train_text_ds,
    val_ds=val_text_ds,
    tokenizer=tokenizer,
    max_source_len=MAX_SOURCE_LEN,
    max_target_len=MAX_TARGET_LEN,
)

print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Tokenized train rows:", len(train_tok))

Map: 100%|██████████| 160/160 [00:00<00:00, 4282.25 examples/s]

Tokenizer vocab size: 2000
Tokenized train rows: 7840


In [7]:
model = build_tiny_t5_model(
    vocab_size=tokenizer.vocab_size,
    max_positions=max(MAX_SOURCE_LEN, MAX_TARGET_LEN),
    d_model=D_MODEL,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {params:,}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=2,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    no_cuda=True,
    report_to=[],
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=collator,
)

Model parameters: 1,306,368


c:\Users\jayan\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\training_args.py:1609: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
C:\Users\jayan\AppData\Local\Temp\ipykernel_20204\4283968085.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [8]:
train_result = trainer.train()
print(train_result)

trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")

eval_metrics = trainer.evaluate()
print("Eval metrics:", eval_metrics)

Step,Training Loss,Validation Loss
200,5.869800,5.755090
400,5.461800,5.322422
600,5.244500,5.052167
800,5.143500,4.879632
1000,5.007100,4.767669
1200,4.846400,4.694137
1400,4.788900,4.635629
1600,4.745000,4.581805
1800,4.694200,4.534855
2000,4.690200,4.499271


TrainOutput(global_step=3920, training_loss=4.856814071110317, metrics={'train_runtime': 723.2892, 'train_samples_per_second': 21.679, 'train_steps_per_second': 5.42, 'total_flos': 8744753074176.0, 'train_loss': 4.856814071110317, 'epoch': 2.0})


Eval metrics: {'eval_loss': 4.31173038482666, 'eval_runtime': 2.5596, 'eval_samples_per_second': 62.509, 'eval_steps_per_second': 15.627, 'epoch': 2.0}


In [10]:
def generate_story_continuation(prompt_prefix: str, max_new_tokens: int = 80) -> str:
    """Generate a continuation for a story prefix.

    This function formats the input prompt with the training task prefix and
    performs stochastic decoding for diverse continuations.

    Args:
        prompt_prefix (str): Beginning of the story provided by the user.
        max_new_tokens (int): Maximum generated token count.

    Returns:
        str: Decoded generated continuation text.
    """
    model.eval()
    prompt = f"continue story: {prompt_prefix.strip()}"
    inputs = tokenizer(prompt, return_tensors="pt")

    # Some fast tokenizers return token_type_ids, but T5 does not use them.
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_k=50,
            num_beams=1,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


sample = generate_story_continuation("Once upon a time there was a tiny dragon")
print(sample)

and a dog with with the . the puppy . on the other , in . they were so , the man and ran to see the little bird ran away . the little little bird . the man was , . the man saw that her big beautiful . when the little an started to ," the next tree with in for a . then , you . but the s . it was very happy and asked that


## Next Tuning Steps

- If training is too slow, reduce `MAX_SAMPLES` to `3000` and `NUM_EPOCHS` to `1`.
- If loss is decreasing and CPU can handle more, increase `MAX_SAMPLES` to `15000`.
- Keep model tiny first; later try `D_MODEL=192`, `NUM_LAYERS=3`.
- For better quality, train longer before increasing architecture size.